<a href="https://colab.research.google.com/github/OrastaRakhmatullayeva/ML-projects/blob/main/KNN_classifier_stars.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [78]:
# from sklearn.impute import  KNNImputer

# x=KNNImputer(n_neighborss=5).fit_transform(X)

In [79]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier


In [80]:
df=pd.read_excel("/content/6_class_csv.xlsx")

In [81]:
df.head()

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Star color,Spectral Class
0,3068,0.002400,0.1700,16.12,0,Red,M
1,3042,0.000500,0.1542,16.60,0,Red,M
2,2600,0.000300,0.1020,18.70,0,Red,M
3,2800,0.000200,0.1600,16.65,0,Red,M
4,1939,0.000138,0.1030,20.06,0,Red,M


In [82]:
df.shape

(240, 7)

In [83]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 240 entries, 0 to 239
Data columns (total 7 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Temperature (K)         240 non-null    int64  
 1   Luminosity(L/Lo)        240 non-null    float64
 2   Radius(R/Ro)            240 non-null    float64
 3   Absolute magnitude(Mv)  240 non-null    float64
 4   Star type               240 non-null    int64  
 5   Star color              240 non-null    object 
 6   Spectral Class          240 non-null    object 
dtypes: float64(3), int64(2), object(2)
memory usage: 13.3+ KB


In [84]:
df['Star type'].value_counts()

,count
Star type,
0,40
1,40
2,40
3,40
4,40
5,40


In [85]:
from sklearn.model_selection import train_test_split

In [86]:
df['Star color'].value_counts()

,count
Star color,
Red,112
Blue,55
Blue-white,26
Blue White,10
yellow-white,8
White,7
Yellowish White,3
Blue white,3
white,3


In [87]:
df["Star color"] = (
    df["Star color"]
    .str.strip()                            # remove leading/trailing spaces
    .str.lower()                            # lowercase everything
    .str.replace("-", " ", regex=False)     # hyphen -> space
    .str.replace(r"\s+", " ", regex=True)   # collapse repeated spaces
)


In [88]:
mapping = {
    "whitish": "white",
    "yellowish": "yellow",
    "yellowish white": "yellow white",
    "white yellow": "yellow white",       # same as yellow white
    "pale yellow orange": "orange",       # your call, see note below
}

df["Star color"] = df["Star color"].replace(mapping)

In [89]:
df['Star color'].value_counts()

,count
Star color,
red,112
blue,56
blue white,41
white,12
yellow white,12
orange,3
yellow,3
orange red,1


In [90]:
df['Spectral Class'].value_counts()

,count
Spectral Class,
M,111
B,46
O,40
A,19
F,17
K,6
G,1


In [91]:
onehot=pd.get_dummies(df['Spectral Class'],prefix='spectral').astype(int)

# print(onehot.head())
df=pd.concat([df,onehot],axis=1)
df.head()

,Temperature (K),Luminosity(L/Lo),Radius(R/Ro),Absolute magnitude(Mv),Star type,Star color,Spectral Class,spectral_A,spectral_B,spectral_F,spectral_G,spectral_K,spectral_M,spectral_O
0,3068,0.002400,0.1700,16.12,0,red,M,0,0,0,0,0,1,0
1,3042,0.000500,0.1542,16.60,0,red,M,0,0,0,0,0,1,0
2,2600,0.000300,0.1020,18.70,0,red,M,0,0,0,0,0,1,0
3,2800,0.000200,0.1600,16.65,0,red,M,0,0,0,0,0,1,0
4,1939,0.000138,0.1030,20.06,0,red,M,0,0,0,0,0,1,0


In [92]:
X=(df.select_dtypes(include='number')).drop(columns='Star type')
y=df['Star type']

In [93]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=00.2,random_state=42,stratify=y)

In [94]:
pipe=Pipeline(
    steps=[
        ("scaler",StandardScaler()),
        ("knn",KNeighborsClassifier(n_neighbors=5))
    ]
)

pipe.fit(X_train,y_train)
print(pipe.score(X_test,y_test))


pipe1=Pipeline(
    steps=[
        ("min",MinMaxScaler()),
        ("knn",KNeighborsClassifier(n_neighbors=5))
    ]
)

pipe1.fit(X_train,y_train)
print(pipe1.score(X_test,y_test))

0.8958333333333334
0.8958333333333334


In [95]:
from sklearn.model_selection import cross_val_score, GridSearchCV,StratifiedKFold

In [96]:
cv=StratifiedKFold(5,shuffle=True,random_state=42)

result=[]
for k in range(1,42,2):
  p=Pipeline([
      ("scalar",StandardScaler()),
      ("knn",KNeighborsClassifier(n_neighbors=k))
  ])
  ball=(cross_val_score(p,X_train,y_train,cv=cv)).mean()

  result.append((k,round(ball, 4)))

print(max(result))

(41, np.float64(0.5417))


In [97]:
grid={
     'knn__n_neighbors':[1,5,11,41],
     "knn__weights":['uniform','distance']
 }

gs=GridSearchCV(pipe,grid,cv=cv,n_jobs=-1)
gs.fit(X_train,y_train)

print(gs.best_params_,
       gs.best_score_)

{'knn__n_neighbors': 1, 'knn__weights': 'uniform'} 0.9790823211875843


In [98]:
print((pipe.get_params().keys()))

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'scaler', 'knn', 'scaler__copy', 'scaler__with_mean', 'scaler__with_std', 'knn__algorithm', 'knn__leaf_size', 'knn__metric', 'knn__metric_params', 'knn__n_jobs', 'knn__n_neighbors', 'knn__p', 'knn__weights'])
